In [2]:
import pennylane as qml
import numpy as np
from NoisyCircuits.QuantumCircuit import QuantumCircuit
from NoisyCircuits.utils import CreateNoiseModel
import os
import pickle

/Users/adam-ukj7r05xnu2fywx/miniconda3/envs/NoisyCircuits/lib/python3.10/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
2026-07-13 17:23:37,247	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [3]:
circuit_list = []
file_path = os.path.join(os.path.expanduser("~"), "NoisyCircuits/noise_models/Sample_Noise_Model_Heron_QPU.csv")
noise_model = CreateNoiseModel(file_path, [["sx", "x", "rx", "rz"], ["cz", "rzz"]]).create_noise_model()
for backend in ["custom", "pennylane", "qiskit", "qulacs"]:
    for fractional in [True, False]:
        circuit_list.append(
            [QuantumCircuit(
                num_qubits=4,
                noise_model=noise_model,
                use_fractional=fractional,
                sim_backend=backend,
                threshold=1e-4,
                verbose=False
            ), fractional]
        )

/Users/adam-ukj7r05xnu2fywx/miniconda3/envs/NoisyCircuits/lib/python3.10/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
/Users/adam-ukj7r05xnu2fywx/miniconda3/envs/NoisyCircuits/lib/python3.10/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
/Users/adam-ukj7r05xnu2fywx/miniconda3/envs/NoisyCircuits/lib/python3.10/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
/Users/adam-ukj7r05xnu2fywx/miniconda3/envs/NoisyCircuits/lib/python3.10/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
 

In [4]:
def fidelity(p:np.ndarray, 
             q:np.ndarray)->np.float64:
    """"
    Helper function that computes the fidelity between two discrete probability distribution by means of the Hellinger Distance.

    Parameters
    ----------
    p : np.ndarray
        Probability distribution of size (n,)
    q : np.ndarray
        Probabiltiy distribution of size (n,)
    
    Returns
    -------
    np.float64
        The Hellinger Distance between the two distributions.

    Raises
    ------
    ValueError
        When the size of p and q don't match
    """
    if p.shape[0] != q.shape[0]:
        raise ValueError("Shape Mismatch between the distributions.")
    fid = 0.0
    for i in range(p.shape[0]):
        fid += (np.sqrt(p[i]) - np.sqrt(q[i]))**2
    fid = np.sqrt(fid) / np.sqrt(2)
    return fid

In [5]:
def ghz_state_noisy():
    for circuit, fractional in circuit_list:
        circuit.refresh()
        circuit.H(0)
        for i in range(circuit.num_qubits-1):
            circuit.CX(i, i+1)
        p_ref = circuit.run_with_density_matrix([0, 1, 2, 3], 1)
        p_mcwf = circuit.execute([0, 1, 2, 3], 1000, 10)
        fid = fidelity(p_ref, p_mcwf)
        if fid > 0.05:
            print(f"Fidelity below limit for {circuit.qpu} QPU with {circuit.sim_backend} backend with Fractional ({fractional}) Gates. Hellinger Distance is {fid:.4f}")

In [6]:
ghz_state_noisy()

2026-07-13 17:26:03,808	INFO worker.py:2007 -- Started a local Ray instance.
/Users/adam-ukj7r05xnu2fywx/miniconda3/envs/NoisyCircuits/lib/python3.10/site-packages/ray/_private/worker.py:2046: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(
2026-07-13 17:26:13,134	INFO worker.py:1839 -- Calling ray.init() again after it has already been called.
2026-07-13 17:26:24,411	INFO worker.py:1839 -- Calling ray.init() again after it has already been called.
2026-07-13 17:26:33,460	INFO worker.py:1839 -- Calling ray.init() again after it has already been called.


(raylet) WARNING: 52 PYTHON worker processes have been started on node: 54079a011fa7774e564fa27b0ff13b9f6542cf5d13c61b0349847007 with address: 137.226.171.5. This could be a result of using a large number of actors, or due to tasks blocked in ray.get() calls (see https://github.com/ray-project/ray/issues/3644 for some discussion of workarounds).


[2026-07-13 17:26:34,411 E 2294556 2298621] core_worker_process.cc:842: Failed to establish connection to the metrics exporter agent. Metrics will not be exported. Exporter agent status: RpcError: Running out of retries to initialize the metrics agent. rpc_code: 14
2026-07-13 17:26:45,134	INFO worker.py:1839 -- Calling ray.init() again after it has already been called.


(raylet) WARNING: 72 PYTHON worker processes have been started on node: 54079a011fa7774e564fa27b0ff13b9f6542cf5d13c61b0349847007 with address: 137.226.171.5. This could be a result of using a large number of actors, or due to tasks blocked in ray.get() calls (see https://github.com/ray-project/ray/issues/3644 for some discussion of workarounds).


2026-07-13 17:26:48,923	INFO worker.py:1839 -- Calling ray.init() again after it has already been called.


(raylet) WARNING: 92 PYTHON worker processes have been started on node: 54079a011fa7774e564fa27b0ff13b9f6542cf5d13c61b0349847007 with address: 137.226.171.5. This could be a result of using a large number of actors, or due to tasks blocked in ray.get() calls (see https://github.com/ray-project/ray/issues/3644 for some discussion of workarounds).
